# AI-01 PC-side AI Environment Check

TRONプロコン2026 AI推論機能のPC側開発環境確認用ノートブック。

## AI input specification
- Sampling rate: 16 kHz
- Channels: mono
- PCM: signed 16-bit
- Input samples: 16000
- Feature shape: 98 x 40
- Model input: [1, 98, 40, 1]
- Classes:
  - 0: OTHER
  - 1: SNORE
  - 2: COUGH

## AI-01 Environment Validation: Result

TRONプロコン2026のAI推論モデル開発用PC環境として、Google Colabを使用する。

### Validated environment
- Google Colab
- Python 3.13.15
- TensorFlow device: CPU

### Main dependencies
- NumPy 2.1.3
- SciPy 1.16.3
- TensorFlow 2.20.0
- scikit-learn 1.6.1
- matplotlib 3.10.0
- pandas 2.2.3

### Validation results
以下のsmoke testが正常に完了したことを確認した。

- NumPy / SciPyによる基本的な信号処理
- TensorFlow / Kerasによるモデル構築・学習・推論
- scikit-learnによる評価指標計算
- requirements.txtの読み込み・依存パッケージ確認

### Known warning
`python -m pip check` 実行時に以下の警告を確認した。

`ipython 7.34.0 requires jedi, which is not installed.`

Colabベース環境由来の警告として扱い、AI開発に必要な処理への影響は確認されていないため、
requirements.txtへのjedi追加は行わない。

### Environment files
- requirements.txt: プロジェクトで直接使用する主要ライブラリのバージョンを固定
- environment-freeze.txt: 動作確認時のColab環境全体を記録

In [3]:
# requirements.txtの動作確認

%pip install -r /content/drive/MyDrive/TRON/AI/requirements.txt

In [1]:
# Python/Colab ランタイム情報の確認

import sys
import platform

print("Python       :", sys.version)
print("Executable   :", sys.executable)
print("Platform     :", platform.platform())

Python       : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Executable   : /usr/bin/python3
Platform     : Linux-6.6.122+-x86_64-with-glibc2.39


In [2]:
# 主要パッケージのバージョン確認

from importlib.metadata import version, PackageNotFoundError

packages = [
    "numpy",
    "scipy",
    "tensorflow",
    "keras",
    "scikit-learn",
    "matplotlib",
    "pandas",
]

for package in packages:
    try:
        print(f"{package:15s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:15s}: NOT INSTALLED")

numpy          : 2.1.3
scipy          : 1.16.3
tensorflow     : 2.20.0
keras          : 3.13.2
scikit-learn   : 1.6.1
matplotlib     : 3.10.0
pandas         : 2.2.3


In [3]:
# importと依存関係の正常性を確認

import numpy as np
import scipy
from scipy import signal
import tensorflow as tf
import sklearn
import matplotlib
import pandas as pd

print("NumPy       :", np.__version__)
print("SciPy       :", scipy.__version__)
print("TensorFlow  :", tf.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("pandas      :", pd.__version__)

print()
print("Devices:")
print(tf.config.list_physical_devices())

NumPy       : 2.1.3
SciPy       : 1.16.3
TensorFlow  : 2.20.0
scikit-learn: 1.6.1
Matplotlib  : 3.10.0
pandas      : 2.2.3

Devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [4]:
!python -m pip check

ipython 7.34.0 requires jedi, which is not installed.


In [5]:
# Numpy/SciPyのsmoke test

FS = 16000
N_SAMPLES = 16000
WINDOW_SIZE = 400
FFT_SIZE = 512

pcm = np.zeros(N_SAMPLES, dtype=np.int16)

# D-07: int16 PCM -> float32
x = pcm.astype(np.float32) / 32768.0

# D-07: periodic Hann
window = signal.windows.hann(WINDOW_SIZE, sym=False)

# 1 frame only for smoke test
frame = x[:WINDOW_SIZE] * window
spectrum = np.fft.rfft(frame, n=FFT_SIZE)
power = np.abs(spectrum) ** 2

assert x.shape == (16000,)
assert x.dtype == np.float32
assert window.shape == (400,)
assert spectrum.shape == (257,)
assert power.shape == (257,)

print("NumPy / SciPy smoke test: OK")

NumPy / SciPy smoke test: OK


In [6]:
# TensorFlow/Kerasのsmoke test

tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(98, 40, 1)),
    tf.keras.layers.Conv2D(
        filters=4,
        kernel_size=(3, 3),
        padding="same",
        activation="relu"
    ),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
)

x_dummy = np.zeros((1, 98, 40, 1), dtype=np.float32)
y_dummy = np.array([0], dtype=np.int32)

loss = model.train_on_batch(x_dummy, y_dummy)
output = model(x_dummy, training=False).numpy()

assert output.shape == (1, 3)
assert np.allclose(np.sum(output, axis=1), 1.0, atol=1e-5)

print("Output:", output)
print("Loss:", loss)
print("TensorFlow / Keras smoke test: OK")

Output: [[0.3337779  0.33311105 0.33311105]]
Loss: 1.0986123
TensorFlow / Keras smoke test: OK


In [7]:
# scikit-learnのsmoke test

from sklearn.metrics import confusion_matrix, f1_score

y_true = np.array([0, 1, 2, 1, 2, 0])
y_pred = np.array([0, 1, 2, 0, 2, 0])

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("Confusion matrix:")
print(cm)

print()
print("Macro F1:", macro_f1)

assert cm.shape == (3, 3)

print("scikit-learn smoke test: OK")

Confusion matrix:
[[2 0 0]
 [1 1 0]
 [0 0 2]]

Macro F1: 0.8222222222222223
scikit-learn smoke test: OK


In [9]:
# requirements.txtの生成

from importlib.metadata import version
from pathlib import Path

direct_dependencies = [
    "numpy",
    "scipy",
    "tensorflow",
    "scikit-learn",
    "matplotlib",
    "pandas",
]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in direct_dependencies
) + "\n"

Path("requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print(requirements)

numpy==2.1.3
scipy==1.16.3
tensorflow==2.20.0
scikit-learn==1.6.1
matplotlib==3.10.0
pandas==2.2.3



In [8]:
import subprocess
import platform
from pathlib import Path

freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)

environment_info = (
    f"Python: {platform.python_version()}\n"
    f"Platform: {platform.platform()}\n\n"
    "=== pip freeze ===\n"
    + freeze
)

Path("environment-freeze.txt").write_text(
    environment_info,
    encoding="utf-8"
)

print("environment-freeze.txt generated.")

environment-freeze.txt generated.
